# 02 · Re-run UNSW-NB15 and NSL-KDD under protocol 1.1

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

UNSW-NB15 and NSL-KDD were built with the protocol 1.0 code. Every other dataset is now on 1.1, so these two are
rebuilt with the same code for consistency, and the cross-dataset overlap (which reads UNSW-NB15) is measured again.
Run every cell **from top to bottom**, one at a time. Nothing to add or edit.

| Step | What it does | Time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests | 2 min |
| 2 | UNSW-NB15 | 10–15 min |
| 3 | NSL-KDD | 2 min |
| 4 | cross-dataset overlap again | 30–60 min |
| 5 | summary table (every row should show the same `code` and `pilot? False`) | 1 min |

Results go to the same places as notebook 01 (Drive and GitHub), and each step pushes before the next starts.


## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `26 passed`)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pathlib, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

import datetime as _dt, os, pathlib, shutil, subprocess

RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')

def git(*args, check=False):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN or '\0', '***').strip()
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def sync(message=None):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    """
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    git('rebase', '--abort'); git('merge', '--abort')      # clear any half-finished earlier pull
    for attempt in range(3):
        git('fetch', REMOTE, 'main', check=True)
        base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
        changed = set(git('diff', '--name-only', base)[1].split())
        untracked = set(git('ls-files', '--others', '--exclude-standard')[1].split())
        # untracked non-record files survive the reset unless GitHub has a file at the same path
        untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
        changed |= untracked
        keep = sorted(f for f in changed if f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        other = sorted(f for f in changed if not f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        stamp = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        hold = pathlib.Path('/content/sync_hold') / stamp if pathlib.Path('/content').exists() else pathlib.Path(REPO_DIR).parent / f'.sync_hold_{stamp}'
        for f in keep:
            (hold / f).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
        if other:
            bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
            for f in other:
                (bak / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
            print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
        git('reset', '--hard', 'FETCH_HEAD', check=True)
        for f in keep:
            dest = pathlib.Path(REPO_DIR, f)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(hold / f, dest)
        if keep:
            git('add', '--', *keep, check=True)
            if git('diff', '--cached', '--quiet')[0] != 0:
                git('commit', '-m', message or 'Save results produced in Colab', check=True)
        code, out = git('push', REMOTE, 'HEAD:main')
        if code == 0:
            head = git('log', '--oneline', '-1')[1]
            print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
            return
        print('GitHub moved while saving; trying again')
    raise SystemExit('could not save to GitHub after 3 tries: copy the messages above into the project thread')


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

def push(message):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records."""
    sync(message)
    print('saved to GitHub:', message)

def run(*args, message):
    """Run one preprocessing command (its output shows below); push the results only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/preprocess.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')


## Step 2 · UNSW-NB15

In [ ]:
run('--dataset', 'unsw_nb15', message='Preprocess UNSW-NB15 (protocol 1.1)')

## Step 3 · NSL-KDD

In [ ]:
run('--dataset', 'nsl_kdd', message='Preprocess NSL-KDD (protocol 1.1)')

## Step 4 · Cross-dataset overlap, CICIDS2017 train vs external targets (re-measured on the 1.1 UNSW-NB15)

In [ ]:
run('--cross-overlap', message='Cross-dataset overlap after UNSW-NB15 1.1 rebuild')

## Step 5 · Summary. Then tell the project thread you are done.

In [ ]:
import json, glob
import pandas as pd
rows = []
for f in sorted(glob.glob('results/data_audit/*/report.json')):
    r = json.load(open(f))
    c = r.get('counts', {})
    rows.append({'dataset': r['dataset'], 'rows read': c.get('rows_read'), 'duplicates removed': c.get('exact_duplicates_removed'),
                 'rows kept': c.get('rows_out', r.get('host_hours')), 'built': r['run']['utc'],
                 'code': (r['run']['git_commit'] or '')[:7], 'pilot?': bool(r.get('max_rows'))})
pd.DataFrame(rows)